In [39]:
!pip install -U "transformers[torch]" datasets evaluate accelerate scikit-learn

In [40]:
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

import evaluate

Carregando dataset

In [41]:
dataset_treino = load_dataset("json", data_files="/content/sample_data/treino.jsonl");
dataset_teste = load_dataset("json", data_files="/content/sample_data/teste.jsonl")

In [42]:
dataset_treino

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 500
    })
})

In [43]:
dataset_teste

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 100
    })
})

Tokenizer para converter string para ids, BERT não recebe strings.

Os input_ids são os IDs dos tokens dentro do vocabulário do BERT.

Por exemplo:

"This movie was amazing"

vira:

["this", "movie", "was", "amazing"]

e depois:

[2023, 3185, 2001, 6429]

Esses números são o que realmente entra no modelo.

In [44]:
model_name = "google-bert/bert-base-uncased"
tokeninzer = AutoTokenizer.from_pretrained(model_name)

Tokeninzando os datasets.

In [45]:
def tokeninzer_function(example):
  return tokeninzer(
      example["prompt"],
      truncation=True,
      padding="max_length",
      max_length=128)

treino_tokeninzaed = dataset_treino.map(tokeninzer_function, batched=True)
teste_tokeninzaed = dataset_teste.map(tokeninzer_function, batched=True)

O BERT precisa de uma propriedade Label 0 ou 1, então vou mapear a propriedade completion para um Label que terá valor 0 ou 1. Também vou tirar as colunas que não serão usadas.

In [46]:
label_map = {
    "venda": 0,
    "suporte": 1
}

def criar_label(example):
    example["label"] = label_map[example["completion"].lower()]
    return example

treino_tokeninzaed = treino_tokeninzaed.map(criar_label)
teste_tokeninzaed = teste_tokeninzaed.map(criar_label)

treino_tokeninzaed = treino_tokeninzaed.remove_columns(["completion", "prompt"])
teste_tokeninzaed = teste_tokeninzaed.remove_columns(["completion", "prompt"])

print(treino_tokeninzaed)
print(teste_tokeninzaed)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'label'],
        num_rows: 500
    })
})
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'label'],
        num_rows: 100
    })
})


Instanciando o BERT e dando significado para os 0 e 1 de acordo com o mapeamento anterior.

In [52]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "venda", 1: "suporte"},
    label2id={"venda": 0, "suporte": 1}
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Configurando métrica de avaliação

In [48]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
  predictions, labels = eval_pred
  predictions = np.argmax(predictions, axis=1)
  return accuracy.compute(predictions=predictions, references=labels)

Configurando treinamento

learning_rate=2e-5: tamanho dos passos utilizados na atualização
dos pesos. Para fine-tuning de BERT,
valores pequenos são comuns.

batch_size=16: quantidade de exemplos processados antes de uma atualização dos parâmetros.

num_train_epochs=3: número de vezes que o modelo percorre o dataset de treinamento.

weight_decay=0.01: regularização que ajuda a reduzir overfitting.

eval_strategy='epoch': executa a avaliação ao final de cada época



In [49]:
training_args = TrainingArguments(
    output_dir="bert_classify_venda_suporte",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
)

Instanciando o Trainer

O Trainer encapsula o loop de treinamento: batches, forward pass, cálculo de loss, backpropagation,
atualização dos parâmetros, avaliação e salvamento de checkpoints.

In [50]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=treino_tokeninzaed["train"],
    eval_dataset=teste_tokeninzaed["train"],
    compute_metrics=compute_metrics
)

trainer.train();

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.151622,1.000000
2,No log,0.052716,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Avaliando modelo

In [53]:
results = trainer.evaluate()
results

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
No log,0.052716,2,1.000000


{'eval_loss': 0.0527162030339241, 'eval_accuracy': 1.0}

In [54]:
teste_results = trainer.predict(teste_tokeninzaed["train"])
teste_results

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


PredictionOutput(predictions=array([[-1.2148399 ,  1.7362034 ],
       [ 1.527138  , -1.1391402 ],
       [-1.1432266 ,  1.7160085 ],
       [-1.1966723 ,  1.8205364 ],
       [-1.1354835 ,  1.7995749 ],
       [-1.1354835 ,  1.7995749 ],
       [ 1.2636182 , -1.8594337 ],
       [ 0.90041006, -1.4828669 ],
       [-1.2219479 ,  1.8058279 ],
       [ 1.1951072 , -1.7655395 ],
       [ 1.1951072 , -1.7655395 ],
       [-1.2384871 ,  1.8433187 ],
       [-1.1966723 ,  1.8205364 ],
       [ 1.0831833 , -1.7765894 ],
       [-1.2349763 ,  1.8228258 ],
       [ 1.2263017 , -1.6578542 ],
       [-1.2168708 ,  1.7981355 ],
       [-1.2168708 ,  1.7981355 ],
       [-1.2168708 ,  1.7981355 ],
       [-1.1354835 ,  1.7995749 ],
       [-1.2148399 ,  1.7362034 ],
       [-1.219471  ,  1.818553  ],
       [-1.2384871 ,  1.8433187 ],
       [ 0.90041006, -1.4828669 ],
       [-1.1432266 ,  1.7160085 ],
       [-1.1996894 ,  1.8341886 ],
       [ 1.0831833 , -1.7765894 ],
       [ 1.2636182 , -1.85

Salvando para uso

In [55]:
trainer.save_model("./bert_classify_venda_suporte_final")
tokeninzer.save_pretrained("./bert_classify_venda_suporte_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./bert_classify_venda_suporte_final/tokenizer_config.json',
 './bert_classify_venda_suporte_final/tokenizer.json')

Uso

In [56]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="./bert_classify_venda_suporte_final",
    tokenizer="./bert_classify_venda_suporte_final"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

O teste abaixo deverá ser classficado com "suporte"

In [57]:
classifier("Quais são os procedimentos para troca do produto?")

[{'label': 'suporte', 'score': 0.9503127336502075}]